## Today {.smaller}

Last week we built convolutional networks that put one label on an image. That
is the only question a classifier can answer, and it is usually not the question
you have.

1. **Transfer learning.** Start from a model someone else trained. Every model
   in the rest of this deck does this, so it comes first.
2. **The four tasks.** Classification, detection, semantic segmentation,
   instance segmentation. What each one takes in and what each one gives back.
3. **Object detection.** Finding and boxing a variable number of objects. The
   two-stage family, then the one-stage family.
4. **Segmentation.** Labelling every pixel, and separating objects of the same
   class.

At the end, a short preview of vision transformers and DETR, which is where
Weeks 8 and 9 pick up.

## Labs and quizzes {.smaller}

**Lab 5** was due yesterday. That was the convolutional network against the
fully connected model from Lab 4, on the same data.

**Quiz 6** is at the end of class. It covers last week's material: image data,
convolution, and CNN architectures. Nothing from today is on it.

**Lab 6** goes out today and is due Wednesday Oct 14. You will fine-tune a
pretrained model and then run a pretrained detector and segmenter, which are the
two things sections 1 and 3 of today are about.

## What a classifier cannot tell you {.smaller}

Here is ResNet-50, trained on ImageNet, applied to the photograph we used all of
last week. Its single most confident answer is **dingo at 0.126**.

That is not a bug. The model was built to return one label for the whole image,
and this image has three animals in it, so the probability mass is spread thin
and the winner is a dog breed.

Three questions it has no way to answer:

- **How many** animals are there?
- **Where** is each one?
- **Which pixels** belong to the cat and which to the dog?

Each of those needs a different output format, and a different head on top of
the same kind of backbone you built last week. That is today.

# Transfer learning

## Transfer learning {.smaller}

**Transfer learning is reusing the weights of a model trained on one task as the
starting point for a different task.**

In vision that almost always means: take a network trained on ImageNet, keep
most of its weights, and attach a new output layer for your own classes.

It works because of what the early layers learn. A convolutional network trained
on ImageNet spends its first layers detecting edges, corners, colour blobs and
textures. None of that is specific to ImageNet's thousand categories. An edge is
an edge whether the photograph is of a dog, a tumour, or a satellite image of a
parking lot.

The later layers are more specialized. They assemble those edges into
ImageNet-specific parts, which is why the layers you replace are the last ones.

::: {.callout-note}
## The practical consequence
You will almost never train a vision model from random weights. Not as a
shortcut, but because starting from pretrained weights is simply better when
your dataset is anything short of enormous.
:::

## Feature extraction and fine-tuning {.smaller}

There are two ways to use pretrained weights, and the difference is which
parameters you let the optimizer change.

| | **Feature extraction** | **Fine-tuning** |
|---|---|---|
| Backbone | frozen, `requires_grad=False` | trainable |
| What trains | the new head only | the head, and some or all of the backbone |
| Learning rate | normal, e.g. `1e-3` | small, e.g. `1e-4` or lower |
| Speed | fast, no backbone gradients | slower |
| Needs | a small dataset is fine | more data, or it overfits |

**Use feature extraction when** your dataset is small and looks like ImageNet
photographs. You are treating the backbone as a fixed feature function.

**Use fine-tuning when** you have enough data, or your images do not look like
ImageNet. Medical scans and satellite imagery are the standard examples: the
edges still transfer, the later layers do not.

The learning rate matters more than it looks. The pretrained weights are already
good. A large learning rate destroys them in the first few steps, before the
randomly initialized head has produced a useful gradient. That failure has a
name, **catastrophic forgetting**, and the fix is a small learning rate, or
freezing the backbone for the first epoch or two.

## Loading a pretrained model

In [1]:
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

# The Weights enum carries the metadata with it, so you never have to guess.
print("categories      ", len(weights.meta["categories"]))
print("parameters      ", f'{weights.meta["num_params"]:,}')
print("reported top-1  ", weights.meta["_metrics"]["ImageNet-1K"]["acc@1"])
print("trained on      ", weights.meta["recipe"])
print()
print("the preprocessing that came with the weights:")
print(weights.transforms())

categories       1000
parameters       11,689,512
reported top-1   69.758
trained on       https://github.com/pytorch/vision/tree/main/references/classification#resnet

the preprocessing that came with the weights:
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


## Replacing the head {.smaller}

A `torchvision` classifier ends in a single `nn.Linear`. For ResNet it is called
`fc`. Swapping it is one assignment, and the new layer is randomly initialized
with `requires_grad=True` by default.

In [2]:
print("original head:", model.fc)

# Freeze everything, then replace the head. Order matters: the new layer is
# created after the freeze, so it keeps requires_grad=True.
for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 10)
print("new head:     ", model.fc)
print()

trainable = [(n, p.numel()) for n, p in model.named_parameters()
             if p.requires_grad]
total = sum(p.numel() for p in model.parameters())
print("still trainable:", trainable)
print(f"training {sum(n for _, n in trainable):,} of {total:,} parameters "
      f"({100 * sum(n for _, n in trainable) / total:.3f}%)")

# Only the trainable ones go to the optimizer. Passing all of them wastes
# memory on optimizer state for parameters that never receive a gradient.
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=1e-3)

original head: Linear(in_features=512, out_features=1000, bias=True)
new head:      Linear(in_features=512, out_features=10, bias=True)

still trainable: [('fc.weight', 5120), ('fc.bias', 10)]
training 5,130 of 11,181,642 parameters (0.046%)


## The input contract {.smaller}

A pretrained model has expectations about its input, and it will not tell you
when you violate them. The ImageNet contract is: three channels in **RGB**
order, resized and cropped to 224x224, converted to float in `[0, 1]`, then
normalized by the ImageNet channel means and standard deviations.

Below, the same weights and the same photograph through five different
pipelines. **All five run without an error.**

[![](images/fig-normalization.png){.dw100}](images/fig-normalization.png){target="_blank" .zoom}

Skipping `Normalize` barely moved the answer, which is why that bug survives for
months. Forgetting to divide by 255 is catastrophic, and the model is
**certain**: "bolo tie" at 1.000. Input scale matters far more than the exact
constants.

## Transfer learning, measured {.smaller}

Two runs on CIFAR-10 with 5,000 training images, 8 epochs, the
same optimizer and the same schedule. Both are ResNet-18. One starts from
ImageNet weights with the backbone frozen; the other starts from random weights
and trains everything.

[![](images/fig-transfer.png){.dw100}](images/fig-transfer.png){target="_blank" .zoom}

**77.3% against 64.8%**, a gap of 12.5 points, while training
5,130 parameters instead of 11,181,642.

The sharper comparison is the first epoch. The frozen model reaches 68.7%
after one epoch, which already beats what the from-scratch model reaches after
8.

## When transfer learning does not help {.smaller}

Three honest limits.

**A large domain gap.** The early layers transfer because edges are universal.
If your inputs are not photographs of the natural world, less transfers.
Spectrograms, medical scans, and remote sensing imagery all still benefit, but
less, and they usually need fine-tuning rather than a frozen backbone.

**Inputs that are not images at all.** Feeding tabular data reshaped into a
square, or a one-channel signal stacked three times, sometimes works and often
does not. The pretrained filters are looking for spatial structure that may not
be there.

**A large enough dataset.** If you have millions of labelled examples in your own
domain, training from scratch catches up and then passes a frozen backbone. The
curve in the previous slide crosses eventually; it just does not cross at
5,000 images.

The useful summary: the smaller your dataset, the more transfer learning buys
you, and nearly everyone's dataset is small.

## Backbone, neck, head {.smaller}

Every model for the rest of this deck is three parts. The vocabulary is worth
fixing now because the diagrams all use it.

**Backbone.** The convolutional network that turns an image into feature maps.
This is a ResNet or similar, pretrained, exactly what you built last week with
the classifier removed.

**Neck.** Optional. Combines or rescales feature maps from several depths of the
backbone before they reach the head. The feature pyramid network later in this
deck is a neck.

**Head.** The task-specific output layers. This is what changes between tasks:
one vector of class scores for classification, boxes and classes for detection,
a full-resolution class map for segmentation.

::: {.callout-note}
## Why this is the right way to see it
Detection and segmentation are not different kinds of network. They are the same
backbone with a different head and a different loss. Once you see that, the
model zoo stops looking like a list of unrelated acronyms.
:::

# The four tasks

## Classification, detection, segmentation {.smaller}

The same photograph, four tasks, four pretrained `torchvision` models. Every
panel below is real model output.

[![](images/fig-four-tasks.png){.dw90}](images/fig-four-tasks.png){target="_blank" .zoom}

## What the label looks like {.smaller}

The pictures show you the output. The part that actually determines the
architecture is the **shape of the target**, which is what the loss compares
against.

| Task | Target for one image | Fixed size? |
|---|---|---|
| **Classification** | one integer class index | yes |
| **Detection** | a list of `(class, x1, y1, x2, y2)` | **no** |
| **Semantic segmentation** | an `(H, W)` integer array, one class per pixel | yes |
| **Instance segmentation** | a set of `(class, box, (H, W) binary mask)` | **no** |

Read the right-hand column again. Classification and semantic segmentation have
targets whose size is known before you see the image. Detection and instance
segmentation do not, because the number of objects is a property of the image.

That one difference is why detection architectures are complicated and
segmentation architectures are not.

## Why detection is harder than classification {.smaller}

A network is a function with a fixed output shape. `nn.Linear(512, 10)` always
produces ten numbers. There is no way to make it produce "however many objects
happen to be in this image".

So every detector works around it, and there are only a few ways to do that.

1. **Propose regions, then classify each one.** Turn a variable number of
   outputs into a variable number of *inputs*, which a fixed-shape network can
   handle one at a time. This is the two-stage family: R-CNN, Fast R-CNN, Faster
   R-CNN.
2. **Predict a fixed grid, then discard.** Emit a fixed, deliberately too-large
   number of boxes, give each one a confidence, and throw away the ones that
   score low or duplicate each other. This is the one-stage family: YOLO, SSD,
   RetinaNet.
3. **Predict a fixed-size set and match it.** Emit exactly 100 slots, let each
   slot be either an object or "nothing", and solve the pairing problem in the
   loss. This is DETR, previewed at the end of today.

Each one also needs a way to measure whether two boxes are the same box, which
is the next slide.

## Intersection over Union {.smaller}

:::: {.columns}
::: {.column width="34%"}
[![](images/iou.png){.dh440}](images/iou.png){target="_blank" .zoom}
:::
::: {.column width="66%"}
**Intersection over Union is the area two boxes share divided by the area they
cover together.**

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

It runs from 0, no overlap at all, to 1, exactly the same box, and it is
scale-free: a value means the same thing for a large box and a small one.

The boxes in the figure are **made up**, with sides in grid cells rather than
pixels, so that 34.4% says nothing about this photograph. Later today we run
the same arithmetic on two real detections.

IoU does three different jobs today, and it helps to keep them apart:

- deciding whether a prediction **counts as correct** (usually IoU > 0.5)
- deciding whether two predictions are **duplicates**, in NMS
- deciding which anchor box is **responsible** for which object, during training
:::
::::

In [3]:
from torchvision.ops import box_iou

# Two boxes the detector really produced on this image, in (x1, y1, x2, y2).
dog = torch.tensor([[602, 295, 1778, 1181]], dtype=torch.float)
cat = torch.tensor([[437, 297, 1730, 1185]], dtype=torch.float)

inter_w = min(dog[0, 2], cat[0, 2]) - max(dog[0, 0], cat[0, 0])
inter_h = min(dog[0, 3], cat[0, 3]) - max(dog[0, 1], cat[0, 1])
inter = (inter_w * inter_h).item()
areas = [((b[0, 2] - b[0, 0]) * (b[0, 3] - b[0, 1])).item() for b in (dog, cat)]

print(f"intersection {inter:>12,.0f} px")
print(f"union        {sum(areas) - inter:>12,.0f} px")
print(f"IoU          {inter / (sum(areas) - inter):>12.3f}")
print(f"box_iou      {box_iou(dog, cat).item():>12.3f}")

intersection      997,152 px
union           1,192,968 px
IoU                 0.836
box_iou             0.836


## Semantic against instance segmentation, measured {.smaller}

Real output on the same photograph: DeepLabV3 left, Mask R-CNN right.

[![](images/fig-semantic-instance.png){.dw92}](images/fig-semantic-instance.png){target="_blank" .zoom}

DeepLabV3 labels 17.2% of the pixels `cat` and 15.5% `dog`,
and the cat pixels come back as **one region of 82,579 pixels**. Both
cats are the same colour in the left panel, because semantic segmentation has
no concept of a cat. It has a concept of cat-ness per pixel.

Mask R-CNN returns **two separate cat masks**, of 302,574 and 229,834
pixels. It also found the bed, which DeepLabV3 could not, because `bed` is not
one of its 21 classes.

::: {.callout-note}
## The distinction in one sentence
Semantic segmentation asks "what class is this pixel"; instance segmentation
asks "which object does this pixel belong to".
:::

# Two-stage detection

## The region proposal idea {.smaller}

The fixed-output-shape problem has an obvious workaround: if a network can only
classify one thing at a time, then give it one thing at a time.

1. **Propose** a set of rectangular regions that might contain an object.
2. **Classify** each region independently, with a normal CNN classifier. Add a
   "background" class so a region containing nothing can say so.
3. **Refine** each box, because the proposal was approximate.
4. **Suppress** duplicates, because many proposals cover the same object.

This is called **two-stage detection**: stage one proposes, stage two classifies
and refines. Everything in this section is a different answer to "how do we do
step 1, and how do we avoid doing step 2 two thousand times".

## R-CNN {.smaller}

:::: {.columns}
::: {.column width="55%"}
[![](images/r-cnn.png){.dw100}](images/r-cnn.png){target="_blank" .zoom}
:::
::: {.column width="45%"}
**R-CNN**, 2014, was the first version of this idea that worked well.

1. Selective search produces about **2,000 region proposals**.
2. Each proposal is **cropped out of the image** and warped to 227x227.
3. Each warped crop runs through AlexNet to get a feature vector.
4. An SVM per class scores the vector, and a separate regressor refines the box.

Say the cost out loud: that is **2,000 full CNN forward passes for one image**.
R-CNN took roughly 47 seconds per image at inference.

It also could not be trained end to end. The CNN, the SVMs and the box
regressors were three separate training stages.
:::
::::

## Selective search {.smaller}

**Selective search** is the algorithm R-CNN used to produce its proposals, and it
is not a neural network at all. It over-segments the image into small regions by
colour and texture, then repeatedly merges the most similar neighbouring regions,
and emits a bounding box for every region it produced along the way.

That gives a few thousand boxes that are likely to align with object boundaries,
which is far better than sliding a window at every position and scale.

Two things to know about it:

- It is **not learned**. The similarity measures are hand-designed, so the
  proposals cannot improve from your data.
- It is **slow**, roughly two seconds per image on a CPU, and it does not run on
  a GPU.

Selective search is the bottleneck that Faster R-CNN removes, which is the point
of the region proposal network a few slides from now.

## Non-maximum suppression {.smaller}

:::: {.columns}
::: {.column width="46%"}
[![](images/non-max-suppression.png){.dw100}](images/non-max-suppression.png){target="_blank" .zoom}
:::
::: {.column width="54%"}
**Non-maximum suppression, or NMS, is the step that removes duplicate
detections of the same object.**

It is needed because many proposals overlap one object, and each of them
produces its own confident box.

The algorithm, for one class:

1. Discard every box whose score is below a **score threshold**.
2. Take the highest-scoring box that is left and **keep** it.
3. Delete every remaining box whose **IoU with the kept box** is above an **NMS
   threshold**, typically 0.5.
4. Repeat from step 2 until no boxes are left.

Two thresholds, and they do different jobs. The score threshold controls how
confident a detection has to be. The NMS threshold controls how much overlap
counts as a duplicate.
:::
::::

## Running a pretrained detector

In [4]:
from torchvision.models.detection import (FasterRCNN_ResNet50_FPN_Weights,
                                          fasterrcnn_resnet50_fpn)
from torchvision.io import read_image

dw = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
det_model = fasterrcnn_resnet50_fpn(weights=dw).eval()
names = dw.meta["categories"]

img_u8 = read_image("images/cats-dog.jpg")
with torch.no_grad():
    pred = det_model([dw.transforms()(img_u8)])[0]

print(f"{len(pred['scores'])} raw detections after the model's own NMS")
print()
for b, l, s in zip(pred["boxes"], pred["labels"], pred["scores"]):
    if s > 0.3:
        print(f"  {names[l]:8s} {s:.3f}  "
              f"[{', '.join(f'{v:.0f}' for v in b)}]")

18 raw detections after the model's own NMS

  cat      0.998  [1493, 312, 1996, 1077]
  cat      0.979  [164, 564, 954, 1151]
  dog      0.959  [602, 295, 1778, 1181]
  bed      0.729  [0, 634, 1881, 1299]
  cat      0.589  [437, 297, 1730, 1185]
  bed      0.488  [5, 955, 2238, 1299]
  couch    0.442  [36, 278, 1740, 1227]
  cat      0.394  [1129, 310, 1873, 1115]


## What the score threshold decides {.smaller}

The model already ran NMS internally. Those 18 boxes are what survived it.
Everything below is the same output, unchanged, viewed at three score thresholds.

[![](images/fig-threshold.png){.dw100}](images/fig-threshold.png){target="_blank" .zoom}

8 detections, then 5, then 4. **The network did not change.** The
only thing that changed is a number you picked after the forward pass.

This is why a detector's reported accuracy is meaningless without its threshold,
and it is why mAP exists. More on that at the end of the next section.

## What NMS removed {.smaller}

[![](images/fig-nms.png){.dw100}](images/fig-nms.png){target="_blank" .zoom}

On the left, every one of the 18 boxes the model emitted. That is the
clutter a detector produces before anything is filtered, and it is why NMS is
not optional.

In the middle, what survives NMS per class plus a 0.5 score threshold:
5 boxes. On the right, the same NMS run **ignoring the class label**:
4 boxes. One box is in the middle panel and not the right one, and
the next slide is about that box.

## A duplicate that survived NMS {.smaller}

At a 0.5 threshold, 5 boxes survive. One of them is wrong in an
interesting way.

[![](images/fig-iou.png){.dw65}](images/fig-iou.png){target="_blank" .zoom}

A `cat` box at 0.589 sits almost exactly on the `dog` box at
0.959. Their IoU is **997,152 / 1,192,968 = 0.836**, which is well
above the usual 0.5 duplicate threshold.

NMS did not remove it because **torchvision applies NMS per class**. The dog box
and the cat box never compete, because they are not the same class. That is the
difference between the last two panels on the previous slide.

Per-class is usually the right default. A person standing in front of a car
genuinely produces two heavily overlapping boxes of different classes, and you
want both. The cost of that default is this slide.

## Fast R-CNN {.smaller}

:::: {.columns}
::: {.column width="56%"}
[![](images/fast-r-cnn-diagram.png){.dw100}](images/fast-r-cnn-diagram.png){target="_blank" .zoom}
:::
::: {.column width="44%"}
**Fast R-CNN**, 2015, fixed the 2,000-forward-passes problem with one
observation: convolution is position-preserving, so a region of the *image*
corresponds to a region of the *feature map*.

So run the backbone **once** over the whole image, then crop the feature map
instead of the pixels.

- one CNN forward pass per image, not 2,000
- the SVMs and separate regressors are replaced by two output heads, so it
  trains end to end with one loss
- inference went from about 47 seconds to about 2 seconds per image

Selective search is still there, and now it is the slow part. Of those two
seconds, nearly all of it is proposal generation.
:::
::::

## ROI pooling {.smaller}

:::: {.columns}
::: {.column width="44%"}
[![](images/roi-pooling.png){.dw100}](images/roi-pooling.png){target="_blank" .zoom}
:::
::: {.column width="56%"}
Cropping the feature map gives you a region of a different size for every
proposal, and the head after it is an `nn.Linear`, which needs a fixed size.

**ROI pooling turns a region of any size into a fixed-size feature map.**

1. Take the proposal's box and project it onto the feature map by dividing the
   coordinates by the backbone's total stride.
2. Divide that region into a fixed grid, usually 7x7.
3. Max-pool within each cell.

Out comes a 7x7 feature map per proposal regardless of the proposal's size, so
the head's input shape is fixed again.

Note step 1 and step 2 both **round to integers**. That rounding is fine for
predicting a box and is a real problem for predicting a mask, which is what ROI
Align fixes at the end of the segmentation section.
:::
::::

## Training a detector {.smaller}

A detector has two outputs per region, so it has two losses, added together.

$$\mathcal{L} = \mathcal{L}_{\text{cls}} + \lambda \, \mathcal{L}_{\text{box}}$$

**The classification loss** is ordinary cross entropy over the classes plus a
background class. Every proposal gets a label, and most proposals are background.

**The box regression loss** is a smooth L1 on the box offsets, and it is applied
**only to proposals that were matched to a real object**. Regressing the box of
a background region is meaningless, so those terms are masked out.

Matching is done with IoU. A proposal with IoU above 0.5 against some ground
truth box is assigned to that object; below 0.3 it is background; in between it
is usually ignored, because the supervision is ambiguous.

This is called a **multi-task loss**, and the pattern recurs. Mask R-CNN adds a
third term for masks. The parts are trained together, and `\lambda` balances them.

## Faster R-CNN {.smaller}

:::: {.columns}
::: {.column width="42%"}
[![](images/faster-r-cnn-overview.png){.dw100}](images/faster-r-cnn-overview.png){target="_blank" .zoom}
:::
::: {.column width="58%"}
**Faster R-CNN**, 2015, replaced selective search with a small convolutional
network that proposes regions: the **region proposal network**, or RPN.

The RPN slides over the same feature map the detection head uses and, at every
position, predicts two things:

- an **objectness score**: is there an object here at all, regardless of class
- **box offsets** that refine a set of predefined anchor boxes

Because the RPN reads the backbone's feature map, proposals cost almost nothing
on top of a forward pass the network was doing anyway. Inference dropped to about
0.2 seconds per image.

More importantly, the proposals are now **learned**. They improve with your
data, which hand-designed selective search could never do, and the whole
detector is one network trained with one loss.

This is the model that produced every box in today's deck.
:::
::::

## Anchors and box offsets {.smaller}

:::: {.columns}
::: {.column width="38%"}
[![](images/anchor-boxes.png){.dw100}](images/anchor-boxes.png){target="_blank" .zoom}
:::
::: {.column width="62%"}
**An anchor box is a box of fixed size and shape, placed at a fixed position on
the feature map, that a prediction is made relative to.**

At each feature map position the RPN places several anchors, typically three
scales times three aspect ratios, so nine. A 50x38 feature map then carries
about 17,000 anchors.

The network does not predict coordinates. It predicts four **offsets** from the
anchor:

$$t_x = \frac{x - x_a}{w_a}, \quad t_y = \frac{y - y_a}{h_a}, \quad
t_w = \log\frac{w}{w_a}, \quad t_h = \log\frac{h}{h_a}$$

Two reasons this parameterization is used rather than raw pixels. Dividing by the
anchor's width and height makes the targets **scale-free**, so the same output
range works for a small object and a large one. The `log` on width and height
keeps the predicted size **positive** no matter what the network outputs, and
makes equal ratios equally far apart.
:::
::::

# One-stage detection

## Skipping the proposal stage {.smaller}

Two-stage detectors are accurate and they do real work twice: once to propose,
once to classify.

**One-stage detectors predict class and box directly from the feature map, in a
single pass, with no proposal step.**

The trick is to fix the output shape in advance. Divide the image into a grid.
Every cell predicts a fixed number of boxes, each with a confidence and a class
distribution, whether or not anything is there. The output tensor's shape then
depends only on the architecture, never on the image.

The cost is a severe class imbalance. Almost every cell contains no object, so
almost every prediction should be "nothing". Early one-stage detectors were
noticeably less accurate than two-stage ones for exactly this reason, and the fix
was a loss function, **focal loss**, that down-weights the easy background
predictions.

## YOLO {.smaller}

:::: {.columns}
::: {.column width="55%"}
[![](images/yolo-overview.png){.dw100}](images/yolo-overview.png){target="_blank" .zoom}
:::
::: {.column width="45%"}
**YOLO**, "You Only Look Once", 2016, was the first one-stage detector to be
both fast and good.

One CNN, one forward pass, boxes and classes straight out of the last layer.
The original ran at 45 frames per second, against Faster R-CNN's 5.

Each grid cell is **responsible** for any object whose centre falls inside it.
That cell predicts the boxes, the confidences, and the class probabilities for
that object.

Its weaknesses came from the same design. A cell could only claim a limited
number of objects, so crowds and small clustered objects were missed, and the
coarse grid made its boxes less precise than a two-stage detector's.

Later versions, and there are many, mostly kept the structure and fixed these:
anchor boxes, multi-scale prediction, and better loss functions.
:::
::::

## The YOLO output tensor {.smaller}

:::: {.columns}
::: {.column width="40%"}
[![](images/yolo-detection-head.png){.dw100}](images/yolo-detection-head.png){target="_blank" .zoom}
:::
::: {.column width="60%"}
The whole output is one tensor of shape

$$S \times S \times (B \cdot 5 + C)$$

- `S x S` is the grid, 7x7 in the original
- `B` is boxes per cell, 2 in the original
- the **5** is `x, y, w, h, confidence` for each box
- `C` is the number of classes, 20 for PASCAL VOC

The `x, y` are the box centre **relative to the cell**, and `w, h` are relative
to the whole image, which is why the numbers all land in `[0, 1]`.
:::
::::

In [5]:
S, B, C = 7, 2, 20
per_cell = B * 5 + C

print(f"per cell   : {B} boxes x 5 + {C} classes = {per_cell} numbers")
print(f"output     : {S} x {S} x {per_cell} = {S * S * per_cell:,} numbers")
print(f"total boxes: {S * S * B} predicted for every image, always")
print()
print("the same shape, as the last layer sees it:")
print(" ", tuple(torch.zeros(1, S, S, per_cell).shape))

per cell   : 2 boxes x 5 + 20 classes = 30 numbers
output     : 7 x 7 x 30 = 1,470 numbers
total boxes: 98 predicted for every image, always

the same shape, as the last layer sees it:
  (1, 7, 7, 30)


## One stage against two {.smaller}

| | **Two-stage** | **One-stage** |
|---|---|---|
| Example | Faster R-CNN | YOLO, SSD, RetinaNet |
| Passes | propose, then classify | one |
| Speed | slower | faster, real time |
| Small objects | better | worse, historically |
| Localization | more precise boxes | less precise boxes |
| Class imbalance | handled by proposal filtering | needs focal loss |

The gap has narrowed a lot. Modern one-stage detectors with a feature pyramid
and focal loss are competitive on accuracy, and they are the default choice
unless you have a specific reason otherwise.

**Choose one-stage** when you have a latency budget: video, anything on a phone
or embedded device, anything live.

**Choose two-stage** when accuracy on small or crowded objects matters more than
speed, and you can afford the compute.

## Feature pyramid networks {.smaller}

:::: {.columns}
::: {.column width="52%"}
[![](images/feature-pyramid-network.png){.dw92}](images/feature-pyramid-network.png){target="_blank" .zoom}
:::
::: {.column width="48%"}
A backbone's feature maps shrink with depth. That creates a problem for
detection that classification never had.

- **Early layers** are high resolution and semantically weak. Good spatial
  precision, poor idea of what the object is.
- **Late layers** are low resolution and semantically strong. They know it is a
  dog, and at 7x7 they cannot say precisely where.

A small object may occupy a few pixels of a late feature map, or none.

**A feature pyramid network builds a set of feature maps that are all
semantically strong, at several resolutions.** It upsamples the deep map and
adds it to the shallower one, repeating up the pyramid.

The head then runs at **every level**: small objects on the high-resolution
levels, large ones on the low-resolution levels. This is the "neck", and it is
in nearly every modern detector including today's Faster R-CNN.
:::
::::

## Detection in practice {.smaller}

**What to reach for.** A pretrained detector from `torchvision.models.detection`
or from the `ultralytics` package, fine-tuned on your data. Training a detector
from scratch is a research project, not a step in a pipeline.

**The API, which you saw a few slides ago.** `fasterrcnn_resnet50_fpn(weights=...)`
in `eval()` mode takes a **list** of image tensors, each one allowed to be a
different size, and returns a **list** of dicts with keys `boxes`, `labels` and
`scores`. In `train()` mode the same model takes images and targets and returns
the loss dict instead.

**mAP, in one paragraph.** Mean average precision is the standard score. For one
class, sweep the score threshold, record precision at each recall level, and
average: that is average precision. Then average over IoU thresholds from 0.5 to
0.95, and finally over classes. The averaging over score thresholds is the point:
it is what makes mAP independent of the threshold choice that the slide earlier
showed changing the answer from 8 detections to 4. The cost is that one
number now hides which threshold you would actually ship.

# Segmentation

## Semantic segmentation {.smaller}

**Semantic segmentation is assigning a class label to every pixel.**

[![](images/semantic-segmentation.png){.dw92}](images/semantic-segmentation.png){target="_blank" .zoom}

The input is an image of shape `(3, H, W)`. The output is an array of shape
`(H, W)` where each entry is a class index. The network actually emits
`(num_classes, H, W)` logits and you take the `argmax` over the class axis,
exactly like classification, run `H * W` times.

So the loss is also exactly classification's loss: cross entropy, averaged over
pixels instead of over a batch of images. The task is not conceptually new.

The **architecture** is new, and for one reason: the output has to be the same
size as the input, and a classifier backbone spends all its effort making things
smaller.

## The resolution problem {.smaller}

Last week every architecture we looked at did the same thing: shrink the spatial
dimensions, grow the channels. That is the right design for classification,
because the answer is one label and spatial detail is noise.

For ResNet-18 on a 224x224 input, the feature map shapes were:

| Stage | Shape | Spatial size |
|---|---|---|
| after `conv1` and pool | `(64, 56, 56)` | 1/4 |
| layer2 | `(128, 28, 28)` | 1/8 |
| layer3 | `(256, 14, 14)` | 1/16 |
| layer4 | `(512, 7, 7)` | 1/32 |

**A segmentation head needs 224x224 back out of a 7x7 feature map.** Each of
those 49 positions is semantically rich and spatially vague; it covers a 32x32
patch of the input and cannot say where within that patch anything was.

Two things are needed. A way to **increase** spatial size inside a network, and a
way to **recover** the fine detail that pooling discarded. The next few slides
are those two things.

## Transposed convolution {.smaller}

**A transposed convolution is a layer that increases spatial size, with learned
weights.**

It works by scattering rather than gathering. An ordinary convolution reads a
patch of the input and writes one output value. A transposed convolution reads
one input value, multiplies the whole kernel by it, and **adds** that into a
patch of the output. Where the patches overlap, contributions sum.

[![](images/transposed-convolution.gif){.dw65}](images/transposed-convolution.gif){target="_blank" .zoom}

It is sometimes called "deconvolution". That name is wrong and worth
un-learning: it does not invert a convolution and does not recover a convolution's
input. It is a convolution whose stride applies to the output instead of the
input, which is where the "transposed" comes from.

## Three ways to upsample {.smaller}

[![](images/fig-upsampling.png){.dw95}](images/fig-upsampling.png){target="_blank" .zoom}

**Nearest neighbour** copies each value into a block. No parameters, no
smoothing, blocky.

**Bilinear** interpolates between neighbours. No parameters, smooth, and it can
never produce a value outside the range of its inputs.

**`ConvTranspose2d`** has weights, so it learns what the upsampled detail should
look like. Here the four kernel weights were set by hand to make the arithmetic
visible; in a real network they are trained.

The output size, which you should be able to predict before running it:

$$H_{\text{out}} = (H_{\text{in}} - 1) \cdot s - 2p + k$$

In [6]:
import torch.nn.functional as F

x = torch.tensor([[[[1., 2.], [3., 4.]]]])          # (1, 1, 2, 2)

print("nearest   ", tuple(F.interpolate(x, scale_factor=2,
                                        mode="nearest").shape))
print("bilinear  ", tuple(F.interpolate(x, scale_factor=2, mode="bilinear",
                                        align_corners=False).shape))

up = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)
print("transposed", tuple(up(x).shape), f"({sum(p.numel() for p in up.parameters())} parameters)")

# (H_in - 1) * s - 2p + k, checked against the layer itself.
for h, k, s, p in [(2, 2, 2, 0), (7, 3, 2, 1), (7, 4, 2, 1), (28, 2, 2, 0)]:
    formula = (h - 1) * s - 2 * p + k
    actual = nn.ConvTranspose2d(1, 1, k, stride=s, padding=p)(
        torch.zeros(1, 1, h, h)).shape[-1]
    print(f"H={h:3d} k={k} s={s} p={p}  ->  formula {formula:3d}  "
          f"actual {actual:3d}  {'ok' if formula == actual else 'MISMATCH'}")

nearest    (1, 1, 4, 4)
bilinear   (1, 1, 4, 4)
transposed (1, 1, 4, 4) (5 parameters)
H=  2 k=2 s=2 p=0  ->  formula   4  actual   4  ok
H=  7 k=3 s=2 p=1  ->  formula  13  actual  13  ok
H=  7 k=4 s=2 p=1  ->  formula  14  actual  14  ok
H= 28 k=2 s=2 p=0  ->  formula  56  actual  56  ok


## The checkerboard artifact {.smaller}

A transposed convolution adds overlapping copies of its kernel into the output.
When the **kernel size is not a multiple of the stride**, different output pixels
receive a different number of contributions, so some are systematically brighter
than others.

[![](images/fig-checkerboard.png){.dw92}](images/fig-checkerboard.png){target="_blank" .zoom}

Each panel above is a constant input through an all-ones kernel, so a correct
upsampler would give a constant output. Kernel 3 with stride 2 does not: the
output ranges from 1 to 4 in a regular grid pattern. That pattern shows up in
generated images and in segmentation masks as a visible checkerboard.

Two fixes, both common. Use **kernel size divisible by the stride**, which is why
`ConvTranspose2d(k=4, stride=2)` and `(k=2, stride=2)` are the pairs you see in
real code. Or avoid the layer entirely: **bilinear upsample, then a normal 3x3
convolution**, which gets you learned weights without the uneven overlap.

## Encoder and decoder {.smaller}

The architecture that solves the resolution problem is two halves.

**The encoder** is a normal convolutional backbone. Downsample, grow channels,
discard spatial detail, end up with a small feature map that knows **what** is in
the image. This is a ResNet, and it is pretrained.

**The decoder** reverses the spatial path. Upsample, shrink channels, end up at
the input resolution with one score per class per pixel. This is where the
transposed convolutions live, and it is trained from scratch.

That alone is not enough, and the reason is worth being precise about. The
encoder genuinely **destroyed** the information the decoder needs. After five
pooling stages, the exact position of a boundary within a 32x32 patch is not in
the feature map in any form, so no decoder, however deep, can recover it. The
output comes out correct in its classes and mushy at its edges.

The fix is to give the decoder a second source of information, which is the next
slide.

## U-Net {.smaller}

[![](images/unet-diagram.png){.dw75}](images/unet-diagram.png){target="_blank" .zoom}

**U-Net**, 2015, was built for biomedical image segmentation and its shape is the
standard encoder-decoder with one addition: the grey horizontal arrows. Each
decoder level receives the feature map from the encoder level at the same
resolution, concatenated onto its own upsampled input.

## Why U-Net has skip connections {.smaller}

A skip connection carries the encoder's feature map at some resolution directly
across to the decoder level at that same resolution.

**The deep path knows what. The skipped path knows where.** The decoder level
receives a semantically strong but spatially vague feature map from below, and a
spatially precise but semantically weak feature map from across. Concatenating
them gives the next convolution both, and it learns to combine them.

Without the skips the class predictions are roughly right and the boundaries are
mush, because boundary information was destroyed by pooling and is only present
in the early layers.

::: {.callout-note}
## Not the same as a residual connection
A **residual** connection, from ResNet last week, **adds** a tensor to the output
of a block a couple of layers later. Its job is to make gradients flow through a
deep network.

A **skip** connection in U-Net **concatenates** across the whole network, from
encoder to decoder. Its job is to restore spatial detail. Same picture, different
operation, different purpose.
:::

## Segmentation losses {.smaller}

The default loss is **per-pixel cross entropy**. The network outputs
`(num_classes, H, W)` logits, the target is `(H, W)` class indices, and
`nn.CrossEntropyLoss` handles exactly that shape with no reshaping needed. It
averages the per-pixel loss over all `H * W` positions.

For two-class segmentation, foreground against background, the usual choice is
one output channel with `nn.BCEWithLogitsLoss` instead.

That is the whole loss, and for balanced problems it is all you need. It has one
failure mode, and it is the common case rather than an edge case: **the average
over pixels is dominated by whatever class has the most pixels.**

A tumour in a scan, a road in a satellite image, a defect on a manufactured part:
the thing you care about is a small fraction of the image, and the loss barely
notices whether you found it. The next slide measures that.

## Class imbalance, measured {.smaller}

A synthetic example, because it makes the arithmetic checkable. One object
occupying **3.22%** of the image, and two predictions.

[![](images/fig-dice.png){.dw85}](images/fig-dice.png){target="_blank" .zoom}

The middle panel is a model that found **nothing**. It scores **96.8%
pixel accuracy**, because 96.8% of the pixels really are background. Its
per-pixel cross entropy is **0.142**, which does not look like failure
either.

Its IoU is **0.000**.

Pixel accuracy is not a useful metric for segmentation, and a per-pixel loss
gives a weak gradient signal when the object is small. The fix for the metric is
to report IoU. The fix for the loss is the next slide.

## Dice loss and the combined loss {.smaller}

**The Dice coefficient measures overlap between a predicted region and a true
region**, and is the segmentation equivalent of IoU.

$$\text{Dice} = \frac{2 |P \cap T|}{|P| + |T|}
= \frac{2 \sum_i p_i t_i}{\sum_i p_i + \sum_i t_i},
\qquad \mathcal{L}_{\text{Dice}} = 1 - \text{Dice}$$

The middle form is the one you implement: multiply the predicted probabilities
by the target elementwise and sum. That form is **differentiable**, which the
set-intersection form is not, so it can be used as a loss directly.

Both Dice and IoU count the same intersection and differ in the denominator, and
they are monotonically related: $\text{Dice} = 2\,\text{IoU} / (1 +
\text{IoU})$. The good prediction above scores Dice 0.918 and IoU
0.848.

Crucially, **neither one contains a term for correctly-predicted background**.
Doing nothing scores 0, no matter how much background there is.

In practice you add them: `loss = BCE + Dice`. Cross entropy gives well-behaved
per-pixel gradients early in training, when the predictions are near 0.5
everywhere and Dice is flat. Dice targets the overlap you are actually scored on.

In [7]:
def dice_loss(logits, target, eps=1.0):
    """Soft Dice. Note what is NOT here: any term for true negatives."""
    p = torch.sigmoid(logits).flatten(1)
    t = target.flatten(1).float()
    inter = (p * t).sum(1)
    return (1 - (2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


# One 64x64 target with a small object.
target = torch.zeros(1, 1, 64, 64)
target[:, :, 30:41, 30:41] = 1
print(f"object is {100 * target.mean():.1f}% of the image")
print()

bce = nn.BCEWithLogitsLoss()
for name, logits in [("finds nothing   ", torch.full((1, 1, 64, 64), -6.0)),
                     ("finds the object", torch.where(target > 0, 6.0, -6.0))]:
    print(f"{name}  BCE {bce(logits, target):.4f}   "
          f"Dice loss {dice_loss(logits, target):.4f}")

object is 3.0% of the image

finds nothing     BCE 0.1797   Dice loss 0.9879
finds the object  BCE 0.0025   Dice loss 0.0401


## Instance segmentation and Mask R-CNN {.smaller}

:::: {.columns}
::: {.column width="55%"}
[![](images/mask-r-cnn-overview.png){.dw100}](images/mask-r-cnn-overview.png){target="_blank" .zoom}
:::
::: {.column width="45%"}
**Instance segmentation is detection plus a mask for each detected object.**

**Mask R-CNN**, 2017, is Faster R-CNN with a third head. For every region the
detector keeps, a small fully convolutional branch predicts a binary mask,
usually 28x28, which is then resized onto the box.

So the multi-task loss gains a term:

$$\mathcal{L} = \mathcal{L}_{\text{cls}} + \mathcal{L}_{\text{box}} +
\mathcal{L}_{\text{mask}}$$

One design detail that matters. The mask branch predicts **one mask per class**
and only the mask for the predicted class contributes to the loss. Masks and
classes do not compete, so the mask branch never has to decide what the object
is. It only has to decide which pixels in this box belong to the object.

This is the model that produced the right-hand panel of the earlier
segmentation slide, with two separate cat masks.
:::
::::

## ROI Align {.smaller}

ROI pooling **rounds** the projected box coordinates to the feature grid. At a
stride of 32, a box edge can move by half a cell, which is 16 input pixels. The
box regression head can correct that. A **mask** cannot be corrected, because
the mask *is* the boundary.

**ROI Align removes the rounding.** It keeps the coordinates as floats, places
fixed sample points in each output cell, and reads the feature map at those
exact positions with **bilinear interpolation** between the four surrounding
values. No quantization anywhere. It is worth several points of mask accuracy,
and it is what `torchvision` uses in its detectors and instance segmenters.

[![](images/fig-roi-align.png){.dw100}](images/fig-roi-align.png){target="_blank" .zoom}

# Vision transformers, a preview

## What convolution assumes {.smaller}

Convolution is built on two assumptions. They are usually right, which is why
everything so far has worked, and it is worth naming them as assumptions.

**Locality.** A pixel's meaning depends on its neighbours. A 3x3 kernel can only
see 3x3, so information from distant parts of the image can only combine after
many layers of stacking.

**Weight sharing, which gives translation equivariance.** The same filter is
applied everywhere, so an edge detector learned at the top left works at the
bottom right. This is what made convolution so much more parameter-efficient
than a fully connected layer last week.

These assumptions are a **prior**: knowledge about images built into the
architecture rather than learned from data. A strong prior is what lets a CNN
learn from a modest dataset.

The question the next few slides preview is what happens if you remove the prior
and let the model learn the relationships instead. The answer: it needs much more
data, and given that data it does better.

## An image as a sequence of patches {.smaller}

[![](images/vit-paper-diagram.png){.dw60}](images/vit-paper-diagram.png){target="_blank" .zoom}

A **Vision Transformer**, or ViT, cuts the image into fixed-size square patches,
flattens each one into a vector, and treats the result as a sequence.

The arithmetic is just reshaping, and it needs no new ideas. A 224x224 image at
patch size 16 gives `(224/16)^2 = 196` patches, each one `16 * 16 * 3 = 768`
numbers. A linear layer projects each patch to the model's width, a learned
**position embedding** is added so the model knows where each patch came from,
and the sequence goes into a standard transformer encoder.

The figure is from the paper and is schematic: it draws nine large patches so
the arrows stay readable, not the 196 a real ViT-Base has.

Nothing above is specific to images except the patching step.

## CNN and ViT compared {.smaller}

:::: {.columns}
::: {.column width="50%"}
[![](images/cnn-vs-vit.png){.dw100}](images/cnn-vs-vit.png){target="_blank" .zoom}
:::
::: {.column width="50%"}
The difference is where the spatial structure comes from.

A **CNN** has locality and weight sharing in the architecture. It cannot help but
respect them.

A **ViT** has neither. Every patch can attend to every other patch from the first
layer, so long-range relationships are available immediately. The cost is that
nothing tells it neighbouring patches are related; it has to learn that, from
data.

That trade shows up as a data requirement. On ImageNet-scale data alone, ViTs
underperform comparable CNNs. Pretrained on a much larger dataset first, they
match or beat them.

**What this means for you.** Use a pretrained ViT or fine-tune one. Training one
from scratch on your own dataset is almost always the wrong call, for exactly
the reason in the paragraph above.
:::
::::

## DETR: detection as set prediction {.smaller}

[![](images/detr-paper-diagram.png){.dw92}](images/detr-paper-diagram.png){target="_blank" .zoom}

Go back to the three ways of handling a variable number of outputs. **DETR**,
2020, is the third one.

A CNN backbone produces feature maps, a transformer encoder-decoder processes
them, and the decoder takes a fixed set of **100 learned object queries**. Each
query outputs one class, including a "no object" class, and one box. Always 100
predictions, for every image.

What is gone is as important as what is there. **No anchor boxes. No NMS.** Both
existed to manage a variable output count, and a fixed set of 100 slots with a
"nothing" option removes the need for either.

## Matching predictions to ground truth {.smaller}

:::: {.columns}
::: {.column width="40%"}
[![](images/bipartite-matching.png){.dw100}](images/bipartite-matching.png){target="_blank" .zoom}
:::
::: {.column width="60%"}
Removing NMS creates a new problem. The 100 predictions come out in no
particular order, and the image has, say, 3 real objects. Which prediction
should be compared against which object?

Pairing them by index would be arbitrary, and would train the network to care
about slot order, which is meaningless.

**DETR pairs predictions with ground truth objects before computing the loss**,
choosing the pairing that minimizes total cost over all possible pairings. The
cost combines class probability and box similarity. This is the assignment
problem, and the **Hungarian algorithm** solves it exactly.

The matched pairs get a classification and box loss. The other 97 predictions
are trained toward "no object".

Because exactly one prediction is responsible for each object, duplicates are
penalized during training rather than removed afterwards. That is why there is
no NMS.
:::
::::

## What Weeks 8 and 9 fill in {.smaller}

Be clear about what just happened. The last four slides used the words
**transformer**, **attention**, **encoder-decoder** and **query** without
defining any of them.

That was on purpose. The *architectural ideas* here, an image as a sequence of
patches and detection as set prediction, do not require knowing how attention
computes anything. The mechanism does, and it is the next two weeks.

**Week 8** introduces sequence models and attention: what a query, a key and a
value are, and what the attention operation actually computes.

**Week 9** builds the transformer from those parts, which is the architecture a
ViT and DETR both use.

When you get there, come back to these four slides. They should read differently.

# Wrap-up

## Wrap-up {.smaller}

**Transfer learning.** Start from pretrained weights, nearly always. Freeze the
backbone and train a new head when your data is small; fine-tune with a small
learning rate when it is not. Measured: 77.3% against 64.8% on
the same architecture and budget.

**The four tasks** differ in the shape of their target, and detection is harder
than classification because its target has no fixed size.

**IoU** does three jobs: scoring a prediction, finding duplicates, and assigning
anchors during training.

**Two-stage detection** proposes regions then classifies them. R-CNN to Fast
R-CNN to Faster R-CNN is one idea being made cheaper three times, ending with
learned proposals from the RPN.

**One-stage detection** predicts a fixed grid in one pass. Faster, and it needs
focal loss to cope with the background imbalance.

**Thresholds are yours, not the model's.** The same output gave 8
detections or 4 depending on a number chosen after the forward pass.

**Segmentation** is classification per pixel. The architecture problem is getting
resolution back, which encoder-decoder plus skip connections solves. Per-pixel
cross entropy fails on small objects; add a Dice term.

**Instance segmentation** is detection plus a mask head, and the mask needs ROI
Align rather than ROI pooling because a boundary cannot survive rounding.

## Next week: sequence models and attention {.smaller}

Everything in these two weeks assumed a **fixed-size grid** of inputs with
**local** structure. Images are the best case for that assumption.

Next week the input is a **sequence**: text, audio, a time series. Three things
change.

1. The length varies from example to example, and the model has to handle that.
2. Position matters, but so does distance. Two words twenty tokens apart can be
   the ones that determine the meaning, and a 3x3 kernel will never see both.
3. The useful relationships are **content-dependent** rather than positional.
   Which earlier word matters depends on what the words are, not on how far away
   they sit.

That last point is what attention is for, and it is where the ViT and DETR
preview from today gets filled in.

## Supplemental content {.smaller}

These are **optional**. No graded work assumes you read them. Today condensed
four weeks of a computer vision course, so this is where the detail went.

:::: {.columns}
::: {.column width="50%"}
**My DSAN 6500 computer vision course**

- [Object detection fundamentals](https://benjaminhoughton.georgetown.domains/computer-vision/week-5/week-5.html)
- [Advanced object detection](https://benjaminhoughton.georgetown.domains/computer-vision/week-6/week-6.html)
- [Image segmentation](https://benjaminhoughton.georgetown.domains/computer-vision/week-7/week-7.html)
- [Vision transformers](https://benjaminhoughton.georgetown.domains/computer-vision/week-8/week-8.html)

Week 5 and 6 go much further on anchor assignment, mAP and the YOLO versions.
Week 7 has the loss function variants and the panoptic segmentation material we
skipped entirely.
:::
::: {.column width="50%"}
**Prof. Hickman's centralized lecture content**

- [Introduction to object detection](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/object-detection/introduction-to-object-detection/notes.html)
- [Advanced object detection](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/object-detection/advanced-object-detection/notes.html)
- [Introduction to image segmentation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/image-segmentation/introduction-to-image-segmentation/notes.html)
- [U-Net summary](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/convolutional-neural-networks/UNet-summary/notes.html)
- [Transfer learning](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/modern-cnn-architectures/transfer-learning/notes.html)
- [Pretrained models in torchvision](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/torch-vision/pretrained-models/notes.html)
- [Introduction to vision transformers](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/vision-transformers/introduction-to-vision-transformers/notes.html)
- [Detection transformers](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/vision-transformers/detection-transformers/notes.html)
:::
::::

<sup>The R-CNN, Fast and Faster R-CNN, ROI pooling, anchor box, YOLO, U-Net, Mask
R-CNN diagrams in this deck are from my DSAN 6500 weeks 5 to 7. The
ViT, DETR and bipartite matching diagrams are from my DSAN 6500 week 8 and the
original papers. Every figure labelled with measured numbers was produced by
running pretrained torchvision models on this deck's photograph.</sup>

## Lab 6 {.smaller}

**Due Wednesday Oct 14.**

Three parts, and the first one is the one that will teach you the most.

1. **Fine-tune a pretrained model.** Take a ResNet-18, replace the head, and
   compare a frozen backbone against fine-tuning the whole thing against
   training from scratch, on the same small dataset. You are reproducing the
   measurement from the transfer learning slide.
2. **Run a pretrained detector.** Faster R-CNN on your own images. Sweep the
   score threshold and see the count change, and find at least one detection
   that is wrong in an interesting way.
3. **Run a pretrained segmenter.** DeepLabV3 and Mask R-CNN on the same image,
   and report the pixel counts that show the semantic against instance
   difference.

No training a detector from scratch. That is deliberate and it is also what you
would do in practice.

## Quiz 6 {.smaller}

**At the end of class today.**

It covers **last week**: image data, the convolution operation, building a CNN,
and the architecture families.

Same four-part format as Quiz 5: true or false, multiple choice, fill in the
blank, and find the bugs in a short script. Twenty-five points, about fifteen
minutes.

Nothing from today is on it. Today's material is on Quiz 7.